# Explainable AI for Pneumonia Detection in Chest X-rays

This notebook runs the complete local pipeline for RSNA pneumonia detection with Grad-CAM, Grad-CAM++, and Integrated Gradients.

In [ ]:
import os
import sys
import json
import random
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == 'notebooks':
    PACKAGE_ROOT = BASE_DIR.parent
elif (BASE_DIR / 'train.py').exists() and (BASE_DIR / 'data').exists():
    PACKAGE_ROOT = BASE_DIR
else:
    PACKAGE_ROOT = BASE_DIR / 'xai_pneumonia'
PROJECT_ROOT = PACKAGE_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = os.environ.get('RSNA_DATA_DIR', '/kaggle/input/rsna-pneumonia-detection-challenge/')
OUTPUT_DIR = PACKAGE_ROOT / 'outputs'
MODEL_PATH = PACKAGE_ROOT / 'model' / 'best_model.h5'
PHASE1_MODEL_PATH = PACKAGE_ROOT / 'model' / 'phase1_model.h5'
HEATMAP_DIR = OUTPUT_DIR / 'heatmaps'
sns.set_style('whitegrid')
print('Project root:', PROJECT_ROOT)
print('Package root:', PACKAGE_ROOT)
print('Data dir:', DATA_DIR)

## 1. Setup & Environment Check

Verify hardware and library versions before running the training or explanation stages.

In [ ]:
import platform
import pydicom
import sklearn
import cv2
import scipy
import skimage

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('OpenCV:', cv2.__version__)
print('pydicom:', pydicom.__version__)
print('scikit-learn:', sklearn.__version__)
print('SciPy:', scipy.__version__)
print('scikit-image:', skimage.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Dataset EDA

Inspect class balance, random image samples, and positive bounding-box examples.

In [ ]:
from xai_pneumonia.data.data_loader import RSNAPneumoniaDataModule

data_module = RSNAPneumoniaDataModule(DATA_DIR, batch_size=32, allow_test_access=True)
metadata = data_module.metadata.copy()
display(metadata.head())

plt.figure(figsize=(6,4))
sns.countplot(x=metadata['label'].map({0:'Normal',1:'Pneumonia'}))
plt.title('Patient-Level Class Distribution')
plt.show()

In [ ]:
sample_ids = data_module.get_random_patient_ids('train', n=16)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for axis, patient_id in zip(axes.flatten(), sample_ids):
    sample = data_module.load_sample(patient_id, augment=False, normalize=False)
    axis.imshow(np.mean(sample['image_rgb'], axis=-1), cmap='gray')
    axis.set_title(f"{patient_id}\nLabel={sample['label']}")
    axis.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
positive_ids = data_module.get_random_patient_ids('train', n=4, label=1, annotated_only=True)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for axis, patient_id in zip(axes.flatten(), positive_ids):
    sample = data_module.load_sample(patient_id, augment=False, normalize=False)
    image = (sample['image_rgb'] * 255).astype(np.uint8).copy()
    for x, y, w, h in sample['bboxes']:
        cv2.rectangle(image, (int(x), int(y)), (int(x+w), int(y+h)), (255, 0, 0), 2)
    axis.imshow(image)
    axis.set_title(patient_id)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 3. Data Split Verification

Confirm that the patient-level stratified split preserves class balance and keeps the test set isolated.

In [ ]:
split_summary = data_module.get_split_summary()
display(split_summary)
plt.figure(figsize=(7,4))
sns.barplot(data=split_summary, x='split', y='positive_ratio')
plt.ylim(0, 1)
plt.title('Positive Ratio by Split')
plt.show()

with open(PACKAGE_ROOT / 'data' / 'splits.json', 'r', encoding='utf-8') as handle:
    splits = json.load(handle)
print({key: len(value) for key, value in splits.items()})

## 4. Preprocessing

Compare raw resized images, ImageNet-standardized tensors, and training-time augmentations.

In [ ]:
patient_id = data_module.get_random_patient_ids('train', n=1)[0]
clean = data_module.load_sample(patient_id, augment=False, normalize=True)
aug = data_module.load_sample(patient_id, augment=True, normalize=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(np.mean(clean['image_rgb'], axis=-1), cmap='gray')
axes[0].set_title('Resized RGB')
axes[1].imshow((clean['image'][:,:,0]), cmap='viridis')
axes[1].set_title('Standardized Channel 0')
axes[2].imshow(np.mean(((aug['image'] * np.array([0.229,0.224,0.225])) + np.array([0.485,0.456,0.406])), axis=-1), cmap='gray')
axes[2].set_title('Augmented View')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Model Architecture

Build the ResNet50-based classifier and inspect the architecture summary.

In [ ]:
from xai_pneumonia.model.resnet50_model import build_resnet50_model

model = build_resnet50_model()
model.summary()
print('Architecture: Input -> ResNet50 backbone -> GAP -> Dropout(0.5) -> Dense(512, relu) -> BatchNorm -> Dropout(0.3) -> Dense(2, softmax)')

## 6. Phase 1 Training (Frozen Backbone)

Run the training script, which performs phase 1, fine-tuning, layer selection, and validation reporting.

In [ ]:
train_cmd = [
    sys.executable,
    str(PACKAGE_ROOT / 'train.py'),
    '--data_dir', DATA_DIR,
    '--output_dir', str(OUTPUT_DIR),
    '--epochs', '25',
    '--batch_size', '32',
]
print(' '.join(train_cmd))
subprocess.run(train_cmd, check=True)

## 7. Phase 2 Fine-Tuning

Load the saved training history and inspect the full frozen-plus-finetuned optimization trace.

In [ ]:
with open(PACKAGE_ROOT / 'model' / 'history.json', 'r', encoding='utf-8') as handle:
    history = json.load(handle)
display(pd.DataFrame(history).head())

## 8. Training Curves

Display the exported 2x2 training curve figure.

In [ ]:
training_curve_path = PACKAGE_ROOT / 'outputs' / 'figures' / 'training_curves.png'
img = plt.imread(training_curve_path)
plt.figure(figsize=(12,8))
plt.imshow(img)
plt.axis('off')
plt.show()

## 9. Grad-CAM Layer Selection

Read the selected target layer and visualize candidate-layer explanations for a validation pneumonia case.

In [ ]:
from xai_pneumonia.model.resnet50_model import load_trained_model
from xai_pneumonia.xai.gradcam import GradCAM
from xai_pneumonia.evaluation.visualizer import plot_gradcam_layer_comparison

trained_model = load_trained_model(str(MODEL_PATH))
val_case = data_module.get_random_patient_ids('val', n=1, label=1, annotated_only=True)[0]
val_sample = data_module.load_sample(val_case, augment=False, normalize=True)
candidate_layers = ['conv3_block4_out', 'conv4_block6_out', 'conv5_block3_out']
layer_overlays = {}
for layer_name in candidate_layers:
    explainer = GradCAM(trained_model, target_layer_name=layer_name)
    _, overlay = explainer.generate(val_sample['image'], val_sample['image_rgb'])
    layer_overlays[layer_name] = overlay
comparison_path = plot_gradcam_layer_comparison(np.mean(val_sample['image_rgb'], axis=-1), layer_overlays, val_case, output_dir=str(OUTPUT_DIR / 'figures'))
print('Saved comparison to', comparison_path)
print('Selected layer:', (PACKAGE_ROOT / 'model' / 'best_gradcam_layer.txt').read_text().strip())

## 10. XAI Method Explanations

Generate case-study explanations for a stratified batch of test images.

In [ ]:
explain_batch_cmd = [
    sys.executable,
    str(PACKAGE_ROOT / 'explain.py'),
    '--data_dir', DATA_DIR,
    '--model_path', str(MODEL_PATH),
    '--mode', 'batch',
    '--output_dir', str(OUTPUT_DIR),
]
subprocess.run(explain_batch_cmd, check=True)

## 11. IG Baseline Ablation

Display one saved ablation figure comparing black, Gaussian-noise, and mean-image baselines.

In [ ]:
ablation_files = sorted((OUTPUT_DIR / 'figures').glob('ig_baseline_comparison_*.png'))
if ablation_files:
    plt.figure(figsize=(16,5))
    plt.imshow(plt.imread(ablation_files[0]))
    plt.axis('off')
    plt.show()
else:
    print('Run the batch or full_test explanation stage to create IG ablation figures.')

## 12. Aggregate XAI Metrics Table

Run full-test heatmap generation followed by locked-test evaluation.

In [ ]:
full_test_cmd = [
    sys.executable,
    str(PACKAGE_ROOT / 'explain.py'),
    '--data_dir', DATA_DIR,
    '--model_path', str(MODEL_PATH),
    '--mode', 'full_test',
    '--output_dir', str(OUTPUT_DIR),
]
subprocess.run(full_test_cmd, check=True)

evaluate_cmd = [
    sys.executable,
    str(PACKAGE_ROOT / 'evaluate.py'),
    '--data_dir', DATA_DIR,
    '--model_path', str(MODEL_PATH),
    '--heatmaps_dir', str(HEATMAP_DIR),
    '--output_dir', str(OUTPUT_DIR),
]
subprocess.run(evaluate_cmd, check=True)

## 13. Statistical Significance Results

Inspect McNemar and Wilcoxon test outputs from the JSON summary.

In [ ]:
with open(OUTPUT_DIR / 'evaluation_summary.json', 'r', encoding='utf-8') as handle:
    summary = json.load(handle)
display(pd.DataFrame(summary['wilcoxon_iou']).T)
display(pd.DataFrame(summary['wilcoxon_stability']).T)
print('McNemar:', summary['mcnemar'])

## 14. Classification Metrics

Display the evaluation confusion matrix and ROC/PR curves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(plt.imread(OUTPUT_DIR / 'figures' / 'confusion_matrix.png'))
axes[0].axis('off')
axes[0].set_title('Confusion Matrix')
axes[1].imshow(plt.imread(OUTPUT_DIR / 'figures' / 'roc_pr_curves.png'))
axes[1].axis('off')
axes[1].set_title('ROC and PR Curves')
plt.tight_layout()
plt.show()

## 15. Failure Mode Analysis

Review saved comparison figures for false negatives and false positives.

In [ ]:
comparison_files = sorted((OUTPUT_DIR / 'figures').glob('xai_comparison_*.png'))[:6]
fig, axes = plt.subplots(len(comparison_files), 1, figsize=(16, 4 * max(1, len(comparison_files))))
if len(comparison_files) == 1:
    axes = [axes]
for axis, file_path in zip(axes, comparison_files):
    axis.imshow(plt.imread(file_path))
    axis.set_title(file_path.name)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 16. Conclusions & Method Recommendation

Use the evaluation summary to recommend the most clinically useful explanation method.

In [ ]:
summary_table = pd.DataFrame(summary['xai_table'])
display(summary_table)
best_iou_row = summary_table[summary_table['Metric'] == 'Mean IoU'].iloc[0]
best_stability_row = summary_table[summary_table['Metric'] == 'SSIM Stability'].iloc[0]
best_iou_method = best_iou_row[['Grad-CAM', 'Grad-CAM++', 'Integrated Gradients']].astype(float).idxmax()
best_stability_method = best_stability_row[['Grad-CAM', 'Grad-CAM++', 'Integrated Gradients']].astype(float).idxmax()
print('Best localization method:', best_iou_method)
print('Best stability method:', best_stability_method)
print('Recommended method: choose the method that balances localization, stability, and runtime in the aggregate table for your manuscript discussion.')